In [3]:
import numpy as np
import pandas as pd
import pickle

In [4]:
ohe_sex = pickle.load(open('ohe_sex.pkl','rb'))
ohe_embarked = pickle.load(open('ohe_embarked.pkl','rb'))
clf = pickle.load(open('clf.pkl','rb'))

---
## Inference Without Pipeline — Manual Reconstruction

This notebook simulates **production inference**: a user provides their data, and we must produce a survival prediction using the model trained in `WithoutPipelines.ipynb`.

**The problem immediately visible:** to serve a single prediction, we must load 3 separate pickle files (`ohe_sex`, `ohe_embarked`, `clf`) and then manually replicate every transformation step from training — in exactly the right order.

### Creating the Raw Input

The input is a 1-row numpy array with the 7 raw feature values in the original column order:
```
index: [0]      [1]     [2]    [3]    [4]   [5]    [6]
value: Pclass  Sex     Age   SibSp  Parch  Fare  Embarked
       2       'male'  31.0   0      0     10.5   'S'
```
`dtype=object` allows mixing integers, floats, and strings in one array.

In [8]:
# Assume user Input
# Pclass/gender/age/SibSp/Parch/Fare/Embarked
test_input = np.array([2,'male',31.0,0,0,10.5,'S'],dtype=object).reshape(1,7)

In [9]:
test_input

array([[2, 'male', 31.0, 0, 0, 10.5, 'S']], dtype=object)

### Encoding Sex

`test_input[:,1]` correctly extracts column index 1 (`'male'`) and passes it to the sex encoder.  
Output `[[0., 1.]]` → female=0, male=1 ✅

In [10]:
test_input_sex = ohe_sex.transform(test_input[:,1].reshape(1,1))

C:\...\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [14]:
test_input_sex

array([[0., 1.]])

### ⚠️ Bug — Wrong Column Index Fed to Embarked Encoder

```python
test_input_embarked = ohe_embarked.transform(test_input[:,1].reshape(1,1))
#                                                          ^^^
#                                                    index 1 = Sex column ('male')
#                                                    Should be index 6 = Embarked ('S')
```

**The bug:** `test_input[:,1]` extracts `'male'` (the Sex value) — but this is being passed to the **Embarked encoder**, which was trained on `['C', 'Q', 'S']`.

**Why doesn't it crash?**  
`handle_unknown='ignore'` was set when the encoder was created. An unknown value (`'male'`) simply gets encoded as all-zeros — no exception is raised.

**The output reveals the bug:**  
`[[0., 0., 0.]]` — all zeros. This means the model receives no information about which port the passenger embarked from.

**The fix:**
```python
test_input_embarked = ohe_embarked.transform(test_input[:,6].reshape(1,1))  # ✅ index 6 = Embarked
# Correct output for 'S': [[0., 0., 1.]]  (C=0, Q=0, S=1)
```

**This bug is the perfect illustration of the core problem with manual preprocessing:** you must track column indices across multiple separate transformers by memory. There is no structural safety net — a wrong index silently produces a wrong result.

In [11]:
test_input_embarked = ohe_embarked.transform(test_input[:,1].reshape(1,1))

In [15]:
test_input_embarked

array([[0., 0., 0.]])

### Extracting Age

`test_input[:,2]` correctly extracts column index 2 (Age = 31.0). Note: no imputation is needed here since the user provided a valid age — but if they hadn't, this step would fail silently (passing `NaN` to the model).

In [12]:
test_input_age = test_input[:,2].reshape(1,1)

### Manual Assembly — Replicating Training Column Order

The transformation pieces are assembled in exactly the same order as during training:
```
test_input[:,[0,3,4,5]]   → [Pclass, SibSp, Parch, Fare]     (4 values)
test_input_age            → [Age]                              (1 value)
test_input_sex            → [female_ohe, male_ohe]            (2 values)
test_input_embarked       → [C_ohe, Q_ohe, S_ohe]            (3 values)  ← BUG: contains sex encoding
                                                         Total: 10 columns
```

This exact column order must match what the `DecisionTreeClassifier` saw during training. There's no validation — if the order is wrong, the model receives a completely different feature vector than expected.

In [13]:
test_input_transformed = np.concatenate((test_input[:,[0,3,4,5]],test_input_age,test_input_sex,test_input_embarked),axis=1)

Shape `(1, 10)` confirms 10 features assembled — matching training. But 3 of those 10 "embarked" columns contain the wrong data (sex encoding) due to the bug above.

In [16]:
test_input_transformed.shape

(1, 10)

### Final Prediction

The model outputs `[1]` (predicted: Survived). However, **this prediction is incorrect** — the model received a corrupted feature vector where the Embarked columns contain the Sex encoding (`[0., 1.]` padded with a zero) instead of the actual Embarked one-hot encoding (`[0., 0., 1.]` for 'S').

The model didn't crash, produced no warning, and returned a confident prediction — but the input was silently wrong throughout.

**This is the fundamental danger of manual inference code:** errors are silent, and the only way to catch them is careful manual review of every index.

In [17]:
clf.predict(test_input_transformed)

array([1])